# 1. Data exploration
## Phase 5 data files
Logging in to Hugging Face, mounting Drive and listing the Phase 5 statute identification files in the dataset.


In [1]:
import os
from huggingface_hub import login, get_token, list_repo_files
from google.colab import drive

login()
# Passing the token on so !python scripts can also download the data
os.environ["HF_TOKEN"] = get_token()

drive.mount('/content/drive')

# Listing every file in the dataset that belongs to Phase 5
all_files = list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")
phase5_files = [f for f in all_files if "Phase_5" in f]

for f in phase5_files:
    print(f)


Mounted at /content/drive
Phase_5_LSI/Test/test.json
Phase_5_LSI/Train/dev.json
Phase_5_LSI/Train/train.json


## Phase 5 files and format
Downloading the train, dev and test files and checking how each case is stored.


In [2]:
# Downloading all the Phase 5 files and looking at how they are stored
import json
from huggingface_hub import hf_hub_download

paths = {}
for f in phase5_files:
    name = f.split("/")[-1]
    paths[name] = hf_hub_download("Exploration-Lab/CS-781-Capstone", f, repo_type="dataset")

data = {}
for name in ["train.json", "dev.json", "test.json"]:
    with open(paths[name]) as f:
        data[name] = json.load(f)

    print()
    print(name, type(data[name]), len(data[name]))
    print(str(data[name])[:600])


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            


train.json <class 'list'> 42750
[{'id': '1000008', 'text': '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of wealth, there has been increase oforganised activities by gangs of anti-socials to extractmoney from affluent sections of society like developers,hoteliers and other businessmen by putting them in fear ofdeath and then to demand substantial sums of money commonlyknown as "Khadani" i.e. protection money. One such gangwas op

dev.json <class 'list'> 10181
[{'id': '1000139', 'text': "Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though the prosecution relied on the evidence of many persons who supposedly witnessedthe occurrence, while deposing in Court, most of them resiled from the stateme

## Cases and statute labels
Counting the cases in each split, how many statute ids each case has and how often each statute id appears.


In [3]:
from collections import Counter

train = data["train.json"]
dev = data["dev.json"]
test = data["test.json"]

print("train keys:", list(train[0].keys()))
print("dev keys:", list(dev[0].keys()))
print("test keys:", list(test[0].keys()))

for name, cases in [("train", train), ("dev", dev)]:
    label_counts = Counter(label for case in cases for label in case["labels"])
    labels_per_case = [len(case["labels"]) for case in cases]

    print()
    print(name, "-", len(cases), "cases,", len(label_counts), "different statute ids")
    print("statute ids per case: min", min(labels_per_case), "max", max(labels_per_case),
          "avg", round(sum(labels_per_case) / len(cases), 2))
    print("most common:", label_counts.most_common(10))
    print("least common:", label_counts.most_common()[-10:])

print()
print("test -", len(test), "cases")
print("unique test ids:", len(set(case["id"] for case in test)))


train keys: ['id', 'text', 'labels']
dev keys: ['id', 'text', 'labels']
test keys: ['id', 'text']

train - 42750 cases, 100 different statute ids
statute ids per case: min 1 max 31 avg 3.71
most common: [(5, 11731), (34, 10380), (97, 7881), (43, 7728), (39, 7121), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
least common: [(15, 336), (27, 329), (74, 319), (50, 318), (59, 310), (92, 309), (66, 309), (23, 304), (25, 287), (84, 272)]

dev - 10181 cases, 100 different statute ids
statute ids per case: min 1 max 28 avg 3.92
most common: [(5, 2936), (34, 2596), (97, 1972), (43, 1932), (39, 1779), (12, 1086), (10, 1033), (13, 1033), (77, 1015), (94, 960)]
least common: [(4, 84), (15, 84), (27, 83), (23, 80), (50, 80), (74, 80), (66, 78), (92, 77), (25, 72), (84, 68)]

test - 13019 cases
unique test ids: 13019


## Rare statutes
Checking which statute ids are in dev but not in train, and how many statutes have only a few train cases.


In [4]:
train_ids = set(label for case in train for label in case["labels"])
dev_ids = set(label for case in dev for label in case["labels"])

print("all statute ids in train:", sorted(train_ids))
print("dev ids missing from train:", sorted(dev_ids - train_ids))

# Counting train statutes by how rare they are
train_counts = Counter(label for case in train for label in case["labels"])
for limit in [300, 500, 1000, 2000]:
    rare = sum(count < limit for count in train_counts.values())
    print(f"statutes with fewer than {limit} train cases: {rare}")


all statute ids in train: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
dev ids missing from train: []
statutes with fewer than 300 train cases: 2
statutes with fewer than 500 train cases: 30
statutes with fewer than 1000 train cases: 57
statutes with fewer than 2000 train cases: 78


## Text casing and document length
Checking how many cases are lowercase, how long the documents are in Phase-1 tokens and which masking tags are used in the text.


In [5]:
from tokenizers import Tokenizer

# Counting cases that are fully lowercase in each split
for name, cases in [("train", train), ("dev", dev), ("test", test)]:
    lowercase = sum(case["text"] == case["text"].lower() for case in cases)
    print(name, "lowercase cases:", lowercase, "of", len(cases))

# Measuring the length of the first 500 cases of each split in Phase-1 tokens
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/CS781/data_p1/tokenizer.json")
for name, cases in [("train", train), ("dev", dev), ("test", test)]:
    lengths = sorted(len(tokenizer.encode(case["text"]).ids) for case in cases[:500])
    print(name, "tokens per case: median", lengths[len(lengths) // 2],
          "90%", lengths[int(0.9 * len(lengths))], "max", lengths[-1])

# Counting how many cases use each masking tag
print()
for tag in ["<SECTION>", "<ACT>", "<ENTITY>"]:
    found = sum(tag in case["text"] for case in train[:500])
    print(f"'{tag}' found in {found} of the first 500 train cases")

print()
print(train[0]["text"][:1500])


train lowercase cases: 6 of 42750
dev lowercase cases: 0 of 10181
test lowercase cases: 1 of 13019
train tokens per case: median 664 90% 4803 max 70432
dev tokens per case: median 858 90% 5134 max 24312
test tokens per case: median 811 90% 3889 max 17878

'<SECTION>' found in 445 of the first 500 train cases
'<ACT>' found in 447 of the first 500 train cases
'<ENTITY>' found in 496 of the first 500 train cases

(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of wealth, there has been increase oforganised activities by gangs of anti-socials to extractmoney from affluent sections of society like developers,hoteliers and other businessmen by putting them in fear ofdeath and then to demand substantial sums of money commonlyknown as "Khadani" i.e. protection money. One such gangwas operating in the

## Chunks needed per case
Checking what share of the cases fit fully in 1, 2, 3 or 4 chunks of 512 tokens, for choosing how many chunks to keep.



In [6]:
# Measuring on 2000 cases of each split, so it doesn't take too long
for name, cases in [("train", train), ("dev", dev), ("test", test)]:
    lengths = [len(tokenizer.encode(case["text"]).ids) for case in cases[:2000]]
    for num_chunks in [1, 2, 3, 4]:
        fits = sum(length <= num_chunks * 512 for length in lengths)
        print(f"{name}: {100 * fits / len(lengths):.1f}% of cases fit in {num_chunks} chunks")
    print()


train: 42.2% of cases fit in 1 chunks
train: 60.3% of cases fit in 2 chunks
train: 68.7% of cases fit in 3 chunks
train: 75.3% of cases fit in 4 chunks

dev: 39.4% of cases fit in 1 chunks
dev: 56.9% of cases fit in 2 chunks
dev: 64.7% of cases fit in 3 chunks
dev: 70.8% of cases fit in 4 chunks

test: 37.9% of cases fit in 1 chunks
test: 55.0% of cases fit in 2 chunks
test: 65.2% of cases fit in 3 chunks
test: 72.8% of cases fit in 4 chunks



## Position of the statute tags in long cases
Counting how many of the `<SECTION>` and `<ACT>` tags are in the first 2048 tokens and in the last 2048 tokens of the long cases.


In [7]:
import re

for name, cases in [("train", train), ("dev", dev), ("test", test)]:
    long_cases = 0
    total_tags, tags_in_first, tags_in_last = 0, 0, 0

    for case in cases[:2000]:
        encoding = tokenizer.encode(case["text"])
        if len(encoding.ids) <= 2048:
            continue
        long_cases += 1

        # Character position where the first 2048 tokens end and where the last 2048 tokens start
        first_end = encoding.offsets[2047][1]
        last_start = encoding.offsets[-2048][0]

        for match in re.finditer(r"<SECTION>|<ACT>", case["text"]):
            total_tags += 1
            if match.start() < first_end:
                tags_in_first += 1
            if match.start() >= last_start:
                tags_in_last += 1

    print(f"{name}: {long_cases} long cases, {total_tags} tags, "
          f"{100 * tags_in_first / total_tags:.1f}% in the first 2048 tokens, "
          f"{100 * tags_in_last / total_tags:.1f}% in the last 2048 tokens")


train: 494 long cases, 11870 tags, 48.3% in the first 2048 tokens, 44.0% in the last 2048 tokens
dev: 584 long cases, 13268 tags, 53.2% in the first 2048 tokens, 43.7% in the last 2048 tokens
test: 545 long cases, 12017 tags, 55.5% in the first 2048 tokens, 49.4% in the last 2048 tokens


# 2. Data preparation
## Making the chunks
Running data_prep.py for cleaning the cases and keeping the first 4 chunks of 512 tokens of each case. A smaller train subset is used, where the rare statutes are kept first and the rest are picked at random.


In [8]:
import os
from huggingface_hub import login, get_token
from google.colab import drive

login()
os.environ["HF_TOKEN"] = get_token()
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase5/data_prep.py \
    --out_dir /content/drive/MyDrive/CS781/data_p5 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Picking the train subset ...
  5263 cases picked to cover the rare statutes
Tokenizing and making chunks ...
  train: 8000 cases, 2.26 chunks per case on average
  dev: 10181 cases, 2.42 chunks per case on average
  test: 13019 cases, 2.41 chunks per case on average

Saved to /content/drive/MyDrive/CS781/data_p5
Rarest statutes in the train subset: [(60, 180), (80, 179), (86, 177), (62, 176), (32, 175), (59, 173), (95, 171), (84, 170), (4, 169), (66, 164)]
Most common statutes in the train subset: [(5, 1997), (97, 1565), (34, 1531), (43, 1353), (39, 1137), (10, 967), (77, 942), (61, 833), (12, 821), (13, 746)]


# 3. Training
## decoder, full fine-tuning
Fine-tuning all the weights of the Phase-1 decoder with the multi-label case head. This model is also used as the teacher for knowledge distillation later.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!python /content/drive/MyDrive/CS781/phase5/train.py \
    --arch decoder --method full \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p5/decoder_full


Mounted at /content/drive
Using device: cuda
8000 train cases, 2000 of 10181 dev cases used for evaluation
16 cases per update, 500 steps per epoch, 1000 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
pos_weight: min 1.73, max 6.91
Backbone loaded, missing keys: 0, unused keys: 1
Method: full, training 111.68M of 111.7M parameters (100.00%)
step 10/1000  epoch 1  loss 0.7542  elapsed 0.2 min
step 20/1000  epoch 1  loss 0.5705  elapsed 0.5 min
step 30/1000  epoch 1  loss 0.5391  elapsed 0.7 min
step 40/1000  epoch 1  loss 0.5118  elapsed 0.9 min
step 50/1000  epoch 1  loss 0.4960  elapsed 1.1 min
step 60/1000  epoch 1  loss 0.5394  elapsed 1.3 min
step 70/1000  epoch 1  loss 0.5127  elapsed 1.6 min
step 80/1000  epoch 1  loss 0.5016  elapsed 1.8 min
step 90/1000  epoch 1  loss 0.5191  elapsed 2.1 min
step 100/1000  epoch 1  loss 0.5122  elapsed 2.4 min
step 110/1000  epoch 1  loss 0.4985  elapsed 2.6 min
step 120/1000  epoch 1  lo

# 4. Prediction
## decoder, full fine-tuning
Choosing the thresholds on the full dev set and then predicting the test cases.


In [2]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/decoder_full \
    --split dev


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/decoder_full/best_checkpoint.pt (step 1000, dev macro-F1 0.2550)
threshold 0.5: macro-F1 0.2472  micro-F1 0.3769  subset accuracy 0.0181  statutes per case 5.18
global threshold 0.40: macro-F1 0.2534  micro-F1 0.3590  subset accuracy 0.0112  statutes per case 8.23
per-statute thresholds: macro-F1 0.2782  micro-F1 0.3716  subset accuracy 0.0125  statutes per case 7.00
Worst 10 statutes (statute, dev cases, F1):
  2  98  0.045
  0  86  0.048
  8  92  0.075
  84  68  0.076
  65  105  0.088
  27  83  0.090
  99  176  0.091
  18  100  0.095
  32  99  0.096
  86  92  0.099
Best 10 statutes (statute, dev cases, F1):
  28  151  0.521
  38  425  0.525
  77  1015  0.540
  5  2936  0.549
  29  125  0.557
  61  932  0.580
  81  346  0.584
  36  142  0.600
  94  958  0.608
  34  2596  0.772
Saved /content/drive/MyDrive/CS781/submissions_p5/decoder_full/thresholds.json



Predicting the test cases and making the submission file.


In [3]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/decoder_full \
    --split test


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/decoder_full/best_checkpoint.pt (step 1000, dev macro-F1 0.2550)
13019 test cases, 6.85 statutes per case on average (per_label thresholds)
Saved /content/drive/MyDrive/CS781/submissions_p5/decoder_full/submission.zip


## convmix, LoRMA fine-tuning
Training only the LoRMA weights inside the Phase-1 convmix layers, the case head is trained fully.


In [4]:
!python /content/drive/MyDrive/CS781/phase5/train.py \
    --arch convmix --method lorma \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p5/convmix_lorma \
    --learning_rate 1e-3


Using device: cuda
8000 train cases, 2000 of 10181 dev cases used for evaluation
16 cases per update, 500 steps per epoch, 1000 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt
pos_weight: min 1.73, max 6.91
Backbone loaded, missing keys: 0, unused keys: 1
Method: lorma, training 2.64M of 107.5M parameters (2.46%)
step 10/1000  epoch 1  loss 0.7557  elapsed 0.2 min
step 20/1000  epoch 1  loss 0.5760  elapsed 0.4 min
step 30/1000  epoch 1  loss 0.5353  elapsed 0.6 min
step 40/1000  epoch 1  loss 0.5102  elapsed 0.8 min
step 50/1000  epoch 1  loss 0.4940  elapsed 0.9 min
step 60/1000  epoch 1  loss 0.5419  elapsed 1.1 min
step 70/1000  epoch 1  loss 0.5140  elapsed 1.3 min
step 80/1000  epoch 1  loss 0.5036  elapsed 1.5 min
step 90/1000  epoch 1  loss 0.5286  elapsed 1.7 min
step 100/1000  epoch 1  loss 0.5259  elapsed 1.8 min
step 110/1000  epoch 1  loss 0.5073  elapsed 2.0 min
step 120/1000  epoch 1  loss 0.5350  elapsed 2.2 min
st

Choosing the thresholds on the dev set.


In [5]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/convmix_lorma \
    --split dev


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/convmix_lorma/best_checkpoint.pt (step 1000, dev macro-F1 0.1103)
threshold 0.5: macro-F1 0.0304  micro-F1 0.1825  subset accuracy 0.0185  statutes per case 1.45
global threshold 0.20: macro-F1 0.1080  micro-F1 0.1810  subset accuracy 0.0000  statutes per case 25.78
per-statute thresholds: macro-F1 0.1297  micro-F1 0.1589  subset accuracy 0.0000  statutes per case 21.83
Worst 10 statutes (statute, dev cases, F1):
  86  92  0.022
  2  98  0.025
  8  92  0.026
  18  100  0.027
  57  115  0.029
  0  86  0.029
  50  80  0.030
  65  105  0.030
  92  77  0.034
  27  83  0.035
Best 10 statutes (statute, dev cases, F1):
  45  707  0.254
  61  932  0.296
  43  1932  0.334
  39  1779  0.338
  35  790  0.351
  77  1015  0.358
  81  346  0.368
  97  1970  0.390
  5  2936  0.464
  34  2596  0.709
Saved /content/drive/MyDrive/CS781/submissions_p5/convmix_lorma/thresholds.json


Predicting the test cases and making the submission file.


In [6]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/convmix_lorma \
    --split test


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/convmix_lorma/best_checkpoint.pt (step 1000, dev macro-F1 0.1103)
13019 test cases, 21.68 statutes per case on average (per_label thresholds)
Saved /content/drive/MyDrive/CS781/submissions_p5/convmix_lorma/submission.zip


## encdec, knowledge distillation
Training the encdec encoder as a student, learning from the gold statutes and from the decoder full fine-tuned model as the teacher.


In [7]:
!python /content/drive/MyDrive/CS781/phase5/distill.py \
    --teacher_ckpt /content/drive/MyDrive/CS781/ckpt_p5/decoder_full/best_checkpoint.pt \
    --p1_ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p5/encdec_distill


Using device: cuda
Student (encdec) backbone loaded, missing keys: 0, unused keys: 68
Teacher: 111.7M parameters, student: 60.8M parameters (54% of the teacher)
16 cases per update, 500 steps per epoch, 1000 steps total
step 10/1000  epoch 1  loss 1.7319  elapsed 0.2 min
step 20/1000  epoch 1  loss 1.4601  elapsed 0.4 min
step 30/1000  epoch 1  loss 1.3759  elapsed 0.6 min
step 40/1000  epoch 1  loss 1.3401  elapsed 0.8 min
step 50/1000  epoch 1  loss 1.3277  elapsed 1.0 min
step 60/1000  epoch 1  loss 1.3616  elapsed 1.2 min
step 70/1000  epoch 1  loss 1.3284  elapsed 1.4 min
step 80/1000  epoch 1  loss 1.3288  elapsed 1.6 min
step 90/1000  epoch 1  loss 1.3442  elapsed 1.8 min
step 100/1000  epoch 1  loss 1.3293  elapsed 2.0 min
step 110/1000  epoch 1  loss 1.3181  elapsed 2.2 min
step 120/1000  epoch 1  loss 1.3298  elapsed 2.4 min
step 130/1000  epoch 1  loss 1.3339  elapsed 2.6 min
step 140/1000  epoch 1  loss 1.3140  elapsed 2.8 min
step 150/1000  epoch 1  loss 1.2919  elapsed 3.

Choosing the thresholds on the dev set.


In [8]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/encdec_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/encdec_distill \
    --split dev


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/encdec_distill/best_checkpoint.pt (step 1000, dev macro-F1 0.1570)
threshold 0.5: macro-F1 0.0650  micro-F1 0.2327  subset accuracy 0.0232  statutes per case 1.94
global threshold 0.25: macro-F1 0.1565  micro-F1 0.2703  subset accuracy 0.0006  statutes per case 11.75
per-statute thresholds: macro-F1 0.1786  micro-F1 0.2437  subset accuracy 0.0002  statutes per case 12.21
Worst 10 statutes (statute, dev cases, F1):
  65  105  0.032
  2  98  0.032
  0  86  0.038
  86  92  0.043
  75  132  0.043
  93  141  0.043
  92  77  0.047
  27  83  0.051
  8  92  0.055
  50  80  0.056
Best 10 statutes (statute, dev cases, F1):
  35  790  0.354
  43  1932  0.362
  39  1779  0.371
  20  122  0.410
  81  346  0.415
  97  1970  0.440
  77  1015  0.462
  5  2936  0.479
  61  932  0.508
  34  2596  0.708
Saved /content/drive/MyDrive/CS781/submissions_p5/encdec_distill/thresholds.json


Predicting the test cases and making the submission file.


In [9]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/encdec_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/encdec_distill \
    --split test


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/encdec_distill/best_checkpoint.pt (step 1000, dev macro-F1 0.1570)
13019 test cases, 11.96 statutes per case on average (per_label thresholds)
Saved /content/drive/MyDrive/CS781/submissions_p5/encdec_distill/submission.zip


## moe, partial fine-tuning
Training only the top layers of the Phase-1 moe model, the lower layers stay frozen.


In [10]:
!python /content/drive/MyDrive/CS781/phase5/train.py \
    --arch moe --method partial \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p5/moe_partial


Using device: cuda
8000 train cases, 2000 of 10181 dev cases used for evaluation
16 cases per update, 500 steps per epoch, 1000 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt
pos_weight: min 1.73, max 6.91
Backbone loaded, missing keys: 0, unused keys: 1
Method: partial, training 71.38M of 196.1M parameters (36.40%)
step 10/1000  epoch 1  loss 0.7590  elapsed 0.2 min
step 20/1000  epoch 1  loss 0.5911  elapsed 0.4 min
step 30/1000  epoch 1  loss 0.5363  elapsed 0.5 min
step 40/1000  epoch 1  loss 0.5144  elapsed 0.7 min
step 50/1000  epoch 1  loss 0.4960  elapsed 0.9 min
step 60/1000  epoch 1  loss 0.5335  elapsed 1.0 min
step 70/1000  epoch 1  loss 0.5081  elapsed 1.2 min
step 80/1000  epoch 1  loss 0.4984  elapsed 1.4 min
step 90/1000  epoch 1  loss 0.5125  elapsed 1.6 min
step 100/1000  epoch 1  loss 0.5076  elapsed 1.7 min
step 110/1000  epoch 1  loss 0.4958  elapsed 1.9 min
step 120/1000  epoch 1  loss 0.5138  elapsed 2.1 min
st

Choosing the thresholds on the dev set.


In [11]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/moe_partial/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/moe_partial \
    --split dev


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/moe_partial/best_checkpoint.pt (step 1000, dev macro-F1 0.1877)
threshold 0.5: macro-F1 0.1517  micro-F1 0.3086  subset accuracy 0.0187  statutes per case 3.67
global threshold 0.35: macro-F1 0.1869  micro-F1 0.3038  subset accuracy 0.0079  statutes per case 8.93
per-statute thresholds: macro-F1 0.2098  micro-F1 0.2989  subset accuracy 0.0083  statutes per case 8.63
Worst 10 statutes (statute, dev cases, F1):
  65  105  0.039
  27  83  0.052
  18  100  0.055
  2  98  0.057
  57  115  0.058
  84  68  0.061
  0  86  0.063
  75  132  0.063
  40  152  0.065
  86  92  0.071
Best 10 statutes (statute, dev cases, F1):
  43  1932  0.402
  61  932  0.410
  14  954  0.425
  28  151  0.427
  97  1970  0.436
  77  1015  0.453
  20  122  0.511
  81  346  0.546
  5  2936  0.554
  34  2596  0.692
Saved /content/drive/MyDrive/CS781/submissions_p5/moe_partial/thresholds.json


Predicting the test cases and making the submission file.


In [12]:
!python /content/drive/MyDrive/CS781/phase5/predict.py \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p5/moe_partial/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p5 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p5/moe_partial \
    --split test


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p5/moe_partial/best_checkpoint.pt (step 1000, dev macro-F1 0.1877)
13019 test cases, 8.35 statutes per case on average (per_label thresholds)
Saved /content/drive/MyDrive/CS781/submissions_p5/moe_partial/submission.zip
